In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
from langchain_groq import ChatGroq
api_key=os.getenv("GROQ_API_KEY")
llm=ChatGroq(groq_api_key=api_key,model="llama-3.1-8b-instant")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000274C1539010>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000274C1770E30>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [8]:
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage
)

In [9]:
speech = """
People across the country, involved in government, political, and social activities, 
are dedicating their time to make the ‘Viksit Bharat Sankalp Yatra’ 
(Developed India Resolution Journey) successful.
"""

In [10]:
speech

'\nPeople across the country, involved in government, political, and social activities, \nare dedicating their time to make the ‘Viksit Bharat Sankalp Yatra’ \n(Developed India Resolution Journey) successful.\n'

In [11]:
chat_message = [
    SystemMessage(content="You are an expert in summarizing speeches."),
    HumanMessage(content=f"Please provide a short and concise summary of the following speech:\nText: {speech}")
]

In [12]:
len(speech.split())

27

In [13]:
response = llm.invoke(chat_message)
response.content

"People across the country are contributing to the 'Viksit Bharat Sankalp Yatra' (Developed India Resolution Journey) through government, political, and social activities."

Prompt Template Text Summarization

In [14]:
from langchain_core.prompts import PromptTemplate

generictemplate = """
Write a summary of the following speech:
Speech: {speech}

Translate the precise summary to {language}
"""

prompt = PromptTemplate(
    input_variables=["speech", "language"],
    template=generictemplate
)

In [15]:
complete_prompt = prompt.format(
    speech=speech,
    language="French"
)

complete_prompt

'\nWrite a summary of the following speech:\nSpeech: \nPeople across the country, involved in government, political, and social activities, \nare dedicating their time to make the ‘Viksit Bharat Sankalp Yatra’ \n(Developed India Resolution Journey) successful.\n\n\nTranslate the precise summary to French\n'

In [16]:
len(complete_prompt.split())

41

In [17]:
chain = prompt | llm

summary = chain.invoke({
    "speech": speech,
    "language": "Hindi"
})

summary.text

"Summary:\nThe 'Viksit Bharat Sankalp Yatra' is gaining momentum as people across the country, involved in government, politics, and social activities, are working together to make this initiative successful.\n\nTranslation to Hindi:\n'Viksit Bharat Sankalp Yatra' की दिशा में देश भर में सरकारी, राजनीतिक और सामाजिक क्षेत्र के लोग मिलकर काम कर रहे हैं जिससे इस पहल को सफल बनाया जा सके।"

Stuff Document Chain Text Summarization

In [18]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
# Load PDF
loader = PyPDFLoader("apjspeech.pdf")
docs = loader.load_and_split()

docs
# Load PDF
loader = PyPDFLoader("apjspeech.pdf")
docs = loader.load_and_split()

docs

[Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'moddate': 'D:20070730160943', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'source': 'apjspeech.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology, \nacademics, arts, literature, business, judiciary, administration, local bodies, farming, \nhome makers, special children, media and above all from the youth and st udent \ncommunity who are the future wealt

In [19]:
template = """Write a concise and short summary of the following speech.

Speech:
{text}
"""

prompt = PromptTemplate(
    input_variables=["text"],
    template=template
)

In [20]:
# Combine document text
text = "\n".join([doc.page_content for doc in docs])

# Create chain
chain = prompt | llm | StrOutputParser()

output_summary = chain.invoke({"text": text})

output_summary

'Here\'s a concise and short summary of the speech by A.P.J. Abdul Kalam:\n\nDr. A.P.J. Abdul Kalam, the 11th President of India, delivered a departing speech after completing five years in office. He reflected on his experiences, highlighting the aspirations of the youth, empowering villages, and mobilizing rural core competence for competitiveness. He emphasized the importance of "Seed to Food" for agricultural growth, overcoming problems through partnership, and courage in combating calamities.\n\nHe also spoke about connectivity for societal transformation, defending the nation, and the youth movement for Developed India 2020. He expressed his confidence in the country\'s ability to become a developed nation before 2020, citing the nation\'s progress in the last 60 years and the power of its 540 million youth.\n\nThe President concluded his speech by thanking the citizens for their love and support, and reiterated his mission to bring connectivity between the hearts and minds of th

Map reduce to Summarize Large documents

In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

final_documents = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=100
).split_documents(docs)

final_documents

[Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'moddate': 'D:20070730160943', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'source': 'apjspeech.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology, \nacademics, arts, literature, business, judiciary, administration, local bodies, farming, \nhome makers, special children, media and above all from the youth and st udent \ncommunity who are the future wealt

In [24]:
len(final_documents)

13

In [25]:
chunks_prompt = """
Please summarize the below speech:
Speech:`{text}`
Summary:
"""

map_prompt_template = PromptTemplate(
    input_variables=["text"],
    template=chunks_prompt
)

In [26]:
final_prompt = """
Provide the final summary of the entire speech with these important points.
Add a Motivation Title. Start the precise summary with an introduction and provide
the summary in numbered points for the speech.

Speech:
{text}
"""

final_prompt_template = PromptTemplate(
    input_variables=["text"],
    template=final_prompt
)

final_prompt_template

PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='\nProvide the final summary of the entire speech with these important points.\nAdd a Motivation Title. Start the precise summary with an introduction and provide\nthe summary in numbered points for the speech.\n\nSpeech:\n{text}\n')

In [27]:
from langchain_core.output_parsers import StrOutputParser

map_chain = map_prompt_template | llm | StrOutputParser()

chunk_summaries = []

for doc in final_documents:
    summary = map_chain.invoke({"text": doc.page_content})
    chunk_summaries.append(summary)

In [28]:
combine_chain = final_prompt_template | llm | StrOutputParser()

combined_text = "\n".join(chunk_summaries)

output = combine_chain.invoke({"text": combined_text})

output

'**Motivation Title:** "Empowering India: A Vision for a Developed Nation"\n\n**Precise Summary with Key Points:**\n\n**Introduction:**\nDr. A.P.J. Abdul Kalam, the former President of India, delivers a farewell speech, reflecting on his experiences and interactions with people from various walks of life. He emphasizes the importance of empowering villages, providing connectivity, and promoting a culture of perseverance and determination.\n\n**Key Messages:**\n\n1. **Accelerate development through the aspirations of the youth**: Kalam emphasizes the potential of India\'s 540 million youth to make a positive impact.\n2. **Empower villages**: He highlights the success of the PURA (Providing Urban Amenities in Rural Areas) program, which has transformed rural communities into self-sufficient entities.\n3. **Mobilize rural core competence for competitiveness**: Kalam stresses the need to tap into rural India\'s potential to drive economic growth.\n4. **Focus on agricultural growth from see

Refine Chain for Summarization

In [29]:
refine_prompt = """
Your job is to produce a final summary.

We already have an existing summary:
{existing_summary}

Now refine the summary using the additional context below.

------------
{text}
------------

Update the summary if needed.
"""

In [30]:
from langchain_core.prompts import PromptTemplate

refine_prompt_template = PromptTemplate(
    input_variables=["existing_summary", "text"],
    template=refine_prompt
)

In [31]:
initial_prompt = """
Write a concise summary of the following speech.

Speech:
{text}
"""

initial_prompt_template = PromptTemplate(
    input_variables=["text"],
    template=initial_prompt
)

In [34]:
from langchain_core.output_parsers import StrOutputParser

initial_chain = initial_prompt_template | llm | StrOutputParser()

refine_chain = refine_prompt_template | llm | StrOutputParser()

In [35]:
# First document summary
summary = initial_chain.invoke({"text": final_documents[0].page_content})

# Refine with remaining documents
for doc in final_documents[1:]:
    summary = refine_chain.invoke({
        "existing_summary": summary,
        "text": doc.page_content
    })

summary

'Refined Summary:\n\nAs the President of India, A.P.J. Abdul Kalam delivered a departing speech emphasizing the importance of achieving a developed India by 2020. His vision was inspired by the aspirations of the youth, with over 15 lakh youth echoing the desire to live in a prosperous, safe, and proud India. He highlighted ten key areas to achieve this goal:\n\n1. Accelerating development through the aspirations of the youth, representing the dreams of 540 million youth nationwide.\n2. Empowering villages, such as the thriving Khuza ma village in Nagaland, where he witnessed a functioning village council with financial powers and saw a prosperous village with fruit and vegetable production.\n3. Mobilizing rural core competence for competitiveness, as seen in the Periyar PURA complex, a 65-village initiative with 300,000 population, providing physical, electronic, and knowledge connectivity, leading to economic connectivity and large-scale employment generation and entrepreneurship cre